# Window-Based Feature Dataset Construction from Imputed Zarr Inputs

## Objective
This notebook constructs the long-format historical feature table used for wildfire classification. The procedure is restricted to label-defined spatial cells and uses imputed feature rasters as the source of daily predictor values.

## Input Source
The sample keys are read from `LABEL/pattern_matches/LABELS_combined.csv`. Each key is defined by `(target_date, row, col)`.

## Historical Window Design
For each key, the notebook extracts the previous 64 days of feature history, excluding the target date itself. The 64-day history is partitioned into four causal windows of length 16 days. These windows are exported as four rows identified by `window_id = 1, 2, 3, 4`.

## Output Structure
The final feature table has the long format:
- `target_date`
- `row`
- `col`
- `window_id`
- feature columns

This representation preserves window identity while keeping all aggregated features for a given window in a single row.


## Environment and Imports

This section loads the required modules and connects the notebook to the local project directories so that the feature-construction utilities can be imported.


In [1]:
# -------------------------
# Imports + module path setup
# -------------------------
from pathlib import Path
import sys
import json
import pandas as pd

BASE = Path('/Users/copperhead07/Computing/DataCollection/RealWork')
UTILS_DIR = BASE / 'UTILS'
FEATURE_DIR = BASE / 'FEATURE'

if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))
if str(FEATURE_DIR) not in sys.path:
    sys.path.insert(0, str(FEATURE_DIR))

from window_feature_builder import WindowFeatureConfig, main


## Configuration

This section defines the label source, the imputed feature directory, the output directory, and the window aggregation rules. The configuration object fully determines how historical windows are constructed and summarized.


In [2]:
# -------------------------
# Config
# -------------------------
LABEL_CSV = BASE / 'LABEL' / 'pattern_matches' / 'LABELS_combined.csv'
FEATURE_ZARR_DIR = BASE / 'FEATURE' / 'imputed_feature_zarr'
OUTPUT_DIR = BASE / 'FEATURE' / 'window_features'

WINDOW_SIZE = 16
NUM_WINDOWS = 4
HISTORY_DAYS = 64
MIN_VALID_DAYS_PER_WINDOW = 1

# Aggregation method per feature.
# Unspecified features use default_agg='mean'.
AGGREGATION_CONFIG = {
    'precip': 'sum',
    'wind_speed': 'mean',
    'tmax': 'mean',
    'tmin': 'mean',
    'rh_max': 'mean',
    'rh_min': 'mean',
    'vpd': 'mean',
    'solar_rad': 'mean',
    'spec_humidity': 'mean',
    'aet': 'mean',
    'pdsi': {'method': 'ema', 'ema_alpha': 0.25},
    'fm100': 'mean',
    'ffmc': 'mean',
    'dmc': 'mean',
    'dc': 'mean',
    'isi': 'mean',
    'bui': 'mean',
    'fwi': 'mean',
    'erc': 'mean',
    'burn_idx': 'mean',
    'ignition_prob_clim': 'mean',
    'elevation': 'mean',
    'slope': 'mean',
    'aspect_sin': 'mean',
    'aspect_cos': 'mean',
}

# If you want to skip specific features, list them here.
EXCLUDE_FEATURES = []

cfg = WindowFeatureConfig(
    base_dir=BASE,
    label_csv=LABEL_CSV,
    feature_zarr_dir=FEATURE_ZARR_DIR,
    output_dir=OUTPUT_DIR,
    window_size=WINDOW_SIZE,
    num_windows=NUM_WINDOWS,
    history_days=HISTORY_DAYS,
    aggregation_config=AGGREGATION_CONFIG,
    default_agg='mean',
    ema_alpha=0.3,
    min_valid_days_per_window=MIN_VALID_DAYS_PER_WINDOW,
    include_features=None,
    exclude_features=EXCLUDE_FEATURES,
    n_jobs=10,
    save_split_files=True,
    save_label_split_files=True,
    drop_incomplete_samples=False,
)

cfg


WindowFeatureConfig(base_dir=PosixPath('/Users/copperhead07/Computing/DataCollection/RealWork'), label_csv=PosixPath('/Users/copperhead07/Computing/DataCollection/RealWork/LABEL/pattern_matches/LABELS_combined.csv'), feature_zarr_dir=PosixPath('/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/imputed_feature_zarr'), output_dir=PosixPath('/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_features'), window_size=16, num_windows=4, history_days=64, aggregation_config={'precip': 'sum', 'wind_speed': 'mean', 'tmax': 'mean', 'tmin': 'mean', 'rh_max': 'mean', 'rh_min': 'mean', 'vpd': 'mean', 'solar_rad': 'mean', 'spec_humidity': 'mean', 'aet': 'mean', 'pdsi': {'method': 'ema', 'ema_alpha': 0.25}, 'fm100': 'mean', 'ffmc': 'mean', 'dmc': 'mean', 'dc': 'mean', 'isi': 'mean', 'bui': 'mean', 'fwi': 'mean', 'erc': 'mean', 'burn_idx': 'mean', 'ignition_prob_clim': 'mean', 'elevation': 'mean', 'slope': 'mean', 'aspect_sin': 'mean', 'aspect_cos': 'mean'}, default_agg='m

## Feature Construction Run

The configured pipeline is executed here. The output paths returned by the builder identify the complete feature table, split-specific files, label-aligned split files, and the run summary.


In [3]:
# -------------------------
# Run pipeline
# -------------------------
out_paths = main(cfg)
print(json.dumps(out_paths, indent=2))


[2026-04-23 20:52:47] INFO | Loaded labels | rows=59710 | dates=3171 | splits=['test', 'train', 'val']
[2026-04-23 20:52:47] INFO | Required samples | rows=59710 | date_min=2012-04-01 | date_max=2023-12-31
[2026-04-23 20:52:47] INFO | Selected 25 features
[2026-04-23 20:52:47] INFO | Feature processing workers: 10
[2026-04-23 20:53:26] INFO | Processed feature ignition_prob_clim | nan_ratio=0.000000
[2026-04-23 20:53:30] INFO | Processed feature ffmc | nan_ratio=0.000000
[2026-04-23 20:53:32] INFO | Processed feature aet | nan_ratio=0.000000
[2026-04-23 20:53:32] INFO | Processed feature burn_idx | nan_ratio=0.000000
[2026-04-23 20:53:32] INFO | Processed feature fm100 | nan_ratio=0.000000
[2026-04-23 20:53:32] INFO | Processed feature dc | nan_ratio=0.000000
[2026-04-23 20:53:32] INFO | Processed feature dmc | nan_ratio=0.000000
[2026-04-23 20:53:33] INFO | Processed feature fwi | nan_ratio=0.000000
[2026-04-23 20:53:33] INFO | Processed feature erc | nan_ratio=0.000000
[2026-04-23 20

{
  "all": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_features/FEATURES_all.csv",
  "train": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_features/FEATURES_train.csv",
  "val": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_features/FEATURES_val.csv",
  "test": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_features/FEATURES_test.csv",
  "labels_train": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_features/LABELS_train.csv",
  "labels_val": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_features/LABELS_val.csv",
  "labels_test": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_features/LABELS_test.csv",
  "missing_stats": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_features/window_feature_missing_stats.csv",
  "feature_list": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_f

## Output Inspection

This section provides a compact inspection of the generated files, including sample rows, feature counts, and summary metadata.


In [4]:
# -------------------------
# Preview outputs
# -------------------------
summary_path = Path(out_paths['summary'])
summary = json.loads(summary_path.read_text())
print(json.dumps(summary['counts'], indent=2))
print('\nSelected features:', len(summary['selected_features']))

x_all = pd.read_csv(out_paths['all'], nrows=10)
print('\nX_all preview:')
print(x_all.head().to_string(index=False))
print('\nColumns:', len(x_all.columns))


{
  "samples_in_labels": 59710,
  "rows_in_long_output": 238840,
  "dropped_incomplete_keys": 0
}

Selected features: 25

X_all preview:
target_date  row  col  window_id     aet      bui  burn_idx       dc      dmc    erc      ffmc   fm100      fwi  ignition_prob_clim      isi      pdsi    precip    rh_max   rh_min  solar_rad  spec_humidity      tmax      tmin      vpd  wind_speed  aspect_cos  aspect_sin  elevation     slope
 2012-04-01    7   46          1 0.52500 1.097193    0.0000 1.280599 1.168764 2.1250 11.455280 3.69375 0.092951            0.083333 0.206068 -0.396650 14.600000 18.075000 8.387500  29.368750       0.000839 53.337500 50.893753 0.059375     0.88125   -0.990392    0.138288  1306.7941  1.656604
 2012-04-01    7   46          2 0.80625 2.553787    3.9375 3.048626 2.641270 6.0625 21.950106 4.03125 1.995573            0.084888 1.871764 -0.597845  1.500000 22.687500 9.424999  46.806250       0.000898 71.093750 67.381256 0.101875     1.15625   -0.990392    0.138288  1306.79

## Split-Level Validation

The split-specific feature files are checked to confirm that each `(target_date, row, col)` key retains the expected number of historical windows.


In [5]:
# -------------------------
# Split sanity checks
# -------------------------
for split in ['train', 'val', 'test']:
    p = Path(out_paths[split])
    if not p.exists():
        continue
    df = pd.read_csv(p)
    n_keys = df[['target_date','row','col']].drop_duplicates().shape[0]
    bad = (df.groupby(['target_date','row','col'])['window_id'].nunique() != cfg.num_windows).sum()
    print(f"{split}: rows={len(df):,} keys={n_keys:,} bad_keys={int(bad):,}")


train: rows=190,120 keys=47,530 bad_keys=0
val: rows=41,160 keys=10,290 bad_keys=0
test: rows=7,560 keys=1,890 bad_keys=0


## Label Alignment Check

The final step performs a simple merge between one feature split and its corresponding label split. This verifies that the long-format feature rows remain correctly aligned with the wildfire supervision table.


In [6]:
# -------------------------
# Optional: merge one split X with labels (example: train)
# -------------------------
x_train = pd.read_csv(out_paths['train'])
y_train = pd.read_csv(out_paths['labels_train'])

# Merge labels at key level, keep long window rows.
xy_train = x_train.merge(y_train, on=['target_date','row','col'], how='inner')
print('xy_train shape:', xy_train.shape)
print(xy_train[['target_date','row','col','window_id','label','type_class','split']].head().to_string(index=False))


xy_train shape: (190120, 32)
target_date  row  col  window_id  label type_class split
 2012-04-01    7   46          1      0   NOFIRE0D train
 2012-04-01    7   46          2      0   NOFIRE0D train
 2012-04-01    7   46          3      0   NOFIRE0D train
 2012-04-01    7   46          4      0   NOFIRE0D train
 2012-04-01   83   73          1      0   NOFIRE0D train
